# DOPF Development

In [1]:
import ams

import datetime

In [2]:
print("Last run time:", datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))

!ams misc --version

Last run time: 2024-01-21 07:49:25
Python   3.9.18
ams      0.8.1
andes    1.8.10.post29+g97528c67
numpy    1.26.2
cvxpy    1.4.1
solvers  CLARABEL, CVXOPT, ECOS,
         ECOS_BB, GLPK, GLPK_MI,
         GUROBI, MOSEK, OSQP, PIQP,
         PROXQP, SCIPY, SCS


In [3]:
ams.config_logger(stream_level=10)

In [4]:
sp = ams.load(ams.get_case('ieee123/ieee123.xlsx'),
              setup=True,
              no_output=True,)

Input format guessed as xlsx.
Parsing input file "/Users/jinningwang/Documents/work/ams/ams/cases/ieee123/ieee123.xlsx"...
Input file parsed in 0.1457 seconds.
Zero line rates detacted in rate_a, rate_b, rate_c, adjusted to 999.
If expect a line outage, please set 'u' to 0.
System set up in 0.0020 seconds.


### Formulation

$
(1): \ \ \ \ \ \min \sum_{i \in N} c_i^p \beta_i = \sum_{i \in N} c_i \Re(S_i + S_i^C)
$

s.t.

$
(2): \ \ \ \ \ P_j = \sum_{k:j \rightarrow k} P_{jk} - \sum_{i:i \rightarrow j}P_{ij}, \forall j \in N
$

$
(3): \ \ \ \ \ q_j = \sum_{k:j \rightarrow k} Q_{jk} - \sum_{i:i \rightarrow j}Q_{ij}, \forall j \in N
$

$
(4): \ \ \ \ \ v_j = v_i - (r_{ij}P_{ij} + x_{ij}Q_{ij}), \forall (i,j) \in B
$

$
(5): \ \ \ \ \ v_i \leq |v_i| \leq \overline{v}_i, \forall i \in N
$

$
(6): \ \ \ \ \ \overline{P}_i^g \geq |P_i^g| \geq \underline{P}_i^g, \forall i \in N
$

$
(7): \ \ \ \ \ \overline{Q}_i^g \geq |Q_i^g| \geq \underline{Q}_i^g, \forall i \in N
$

$
(8): \ \ \ \ \ l_{ij} \leq \overline{I}_{ij}, \forall (i,j) \in B
$

Decision variables: $ P_{ij}, Q_{ij}, l_{ij}, v_i, q_i $

Question: Eqn (8) is missing?

## Development

1. Eqn (2) -> Constr ``pb``: need to include ``r_ij``
2. Eqn (3) -> Constr ``qb``: need to include ``x_ij``; should be similar to ``pb``?
3. Eqn (4) -> Constr ``lvd``

In [5]:
sp.DOPF.init(no_code=True)

DOPF data check passed.
Built system matrices in 0.0026 seconds.
- Generating symbols for DOPF
Parse Params in 0.0033 seconds
Parse Vars in 0.0005 seconds
Parse Constr <pglb>: -pg + mul(nctrle, pg0) + mul(ctrle, pmin)  <= 0
Parse Constr <pgub>: pg - mul(nctrle, pg0) - mul(ctrle, pmax)  <= 0
Parse Constr <pb>: Bbus@aBus + Pbusinj + Cl@pd + Csh@gsh - Cg@pg  == 0
Parse Constr <plflb>: -Bf@aBus - Pfinj - rate_a  <= 0
Parse Constr <plfub>: Bf@aBus + Pfinj - rate_a  <= 0
Parse Constr <alflb>: -CftT@aBus - amax  <= 0
Parse Constr <alfub>: CftT@aBus - amax  <= 0
Parse Constr <qglb>: -qg + mul(ug, qmin)  <= 0
Parse Constr <qgub>: qg - mul(ug, qmax)  <= 0
Parse Constr <vu>: vsq - vmax**2  <= 0
Parse Constr <vl>: -vsq + vmin**2  <= 0
Parse Constr <lvd>: CftT@vsq - (r * plf + x * qlf)  == 0
Parse Constr <qb>: sum(qd) - sum(qg)  == 0
Parse Constrs in 0.0120 seconds
Parse Objective <obj>: MIN. sum(mul(c2, power(pg, 2)))+ sum(mul(c1, pg))+ sum(mul(ug, c0))
Parse Objective in 0.0007 seconds
Finalize: 

True

In [6]:
sp.DOPF.run(solver='PIQP')

DOPF has already been initialized.
DOPF solved as optimal in 0.0530 seconds, converged after 9 iterations using solver PIQP.


True